# Model Atlas Tool argument audit

Read-only companion to the September 9, 2026 review. Run with Python 3.12+ from anywhere inside the repository. Uses only the standard library. The 72 saved observations are configuration x case x trial, not independent model tests. V2 eligibility is not measured execution success. The later 48-result runtime diagnostic is a separate cohort.


In [ ]:
import hashlib
import json
import sqlite3
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend/app").is_dir())
evidence = root / "artifacts/reference-workload"
source = evidence / "tool-selection-challenge-v20.json"
assert hashlib.sha256(source.read_bytes()).hexdigest() == "cb3a7e6a6ee873d969ce096a25072461f87cd102667305d1b4369ffecc5b4d24"
audit = json.loads((evidence / "tool-argument-contract-audit-v21.json").read_text(encoding="utf-8"))
rows = audit["rows"]
assert len(rows) == len({(r["entry"], r["id"], r["trial"]) for r in rows}) == 72
assert sum(r["v1_normalized_execution_success"] for r in rows) == 56
assert sum(r["previous_success_now_blocked"] for r in rows) == 46
assert sum(r["v2_execution_allowed"] for r in rows) == 15
assert sum(r["v2_selected_and_allowed"] for r in rows) == 10
print(audit["summary"])


In [ ]:
connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row
connection.execute("CREATE TABLE audit_rows (entry TEXT, v1_normalized_execution_success INTEGER, v2_execution_allowed INTEGER)")
connection.executemany(
    "INSERT INTO audit_rows VALUES (?, ?, ?)",
    [(r["entry"], r["v1_normalized_execution_success"], r["v2_execution_allowed"]) for r in rows],
)
query = "WITH cohort AS (\n  SELECT CASE entry\n    WHEN 'small-baseline' THEN '0.5B 기본'\n    WHEN 'medium-candidate' THEN '1.5B 후보'\n    ELSE '1.5B 프롬프트 변형' END AS configuration,\n    CASE WHEN v2_execution_allowed = 1 THEN 'v2 허용' ELSE 'v2 차단' END AS decision\n  FROM audit_rows\n  WHERE v1_normalized_execution_success = 1\n)\nSELECT configuration, decision, COUNT(*) AS observations,\n       SUM(COUNT(*)) OVER (PARTITION BY configuration) AS denominator,\n       24 AS all_trials\nFROM cohort\nGROUP BY configuration, decision\nORDER BY configuration, decision"
chart_rows = [dict(row) for row in connection.execute(query)]
connection.close()
artifact_path = root / "docs/reports/2026-09-09_tool_argument_review/artifact.json"
artifact = json.loads(artifact_path.read_text(encoding="utf-8"))
expected = [{k: r[k] for k in chart_rows[0]} for r in artifact["snapshot"]["datasets"]["prior_success"]]
sort_key = lambda r: (r["configuration"], r["decision"])
assert sorted(chart_rows, key=sort_key) == sorted(expected, key=sort_key)
assert sum(r["observations"] for r in chart_rows) == 56
print(json.dumps(chart_rows, ensure_ascii=False, indent=2))


In [ ]:
runtime = json.loads((evidence / "runtime-matrix-diagnostic-tool-argument-v21.json").read_text(encoding="utf-8"))
trace = json.loads((evidence / "tool-argument-runtime-audit-v21.json").read_text(encoding="utf-8"))
assert sum(e["result_count"] for e in runtime["entries"]) == len(trace["rows"]) == 48
assert sum(e["tool_execution_summary"]["successful_call_count"] for e in runtime["entries"]) == 5
blocked = [r for r in trace["rows"] if r["guard"]["execution_allowed"] is False]
assert len(blocked) == 36
assert all(s["attempt_count"] == 0 for r in blocked for s in r["steps"])
assert all(r["guard"]["raw_arguments_hash"] == r["guard"]["normalized_arguments_hash"] for r in trace["rows"])
assert sum(all(s["attempt_count"] == 0 for s in r["steps"]) for r in trace["rows"]) == 43
print(trace["summary"])
print("All evidence and chart reconciliation checks passed.")


In [ ]:
connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row
for source_id, dataset, parameter, document in [
    ("audit", "offline", "audit_json", audit),
    ("runtime", "runtime", "runtime_json", runtime),
]:
    spec = next(s for s in artifact["manifest"]["sources"] if s["id"] == source_id)
    computed = [dict(r) for r in connection.execute(spec["query"]["sql"], {parameter: json.dumps(document)})]
    expected = [{k: r[k] for k in computed[0]} for r in artifact["snapshot"]["datasets"][dataset]]
    assert sorted(computed, key=lambda r: r["configuration"]) == sorted(expected, key=lambda r: r["configuration"])
connection.close()
print("Both report tables match executed SQLite queries.")


## Interpretation boundaries

The old compiler changed 53 outputs, but the 46 previously successful calls now blocked are not all proven old grading errors: authorization and document membership rules changed too. Historical model prompts had no trusted document catalog. New runtime recovery tests fail under disabled failure simulation and unchanged evaluator requirements; do not interpret that as an isolated model regression. The official Gate is still BLOCKED, and these diagnostics do not promote evidence.
